# Notebook to Create Torchscript Model from Raw Weights using superdopnet Class

This notebook downloads the raw weights and uses the `SuperdropNet` class in `superdropnet.py` to create the final torchscript model. The torchscript models used for the paper are also provided in the github repository:
- `superdropnet/superdropnet_final_no_constraints.pt`
- `superdropnet/superdropnet_final_sharma2025.pt`
- `superdropnet/superdropnet_final_sharma2025_60s.pt`

In case you only want to use these models, you do not need to execute this notebook.

In [ ]:
import sys
import numpy as np
import os

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
torch.__version__

'2.5.1'

# Download Model Weights

In [4]:
import requests
import tqdm


def download_checkpoint(filename, url='https://share.hereon.de/index.php/s/kgsExSCm5Pck8Sn/download/superdropnet_final.ckpt'):
    """
    Downloads checkpoint from a URL with a progress bar.

    Args:
        url: The URL of the file to download.
        filename: The name of the file to save the downloaded content to.
    """
    
    response = requests.get(url, stream=True)
    response.raise_for_status()  # Raise an exception for bad status codes

    total_size = int(response.headers.get("content-length", 0))
    block_size = 1024  # 1KB

    print("Downloading checkpoint", url)

    progress_bar = tqdm.tqdm(total=total_size, unit="iB", unit_scale=True)

    with open(filename, "wb") as file:
        for data in response.iter_content(block_size):
            progress_bar.update(len(data))
            file.write(data)

    progress_bar.close()

    if total_size != 0 and progress_bar.n != total_size:
        print("ERROR, something went wrong")


In [ ]:
download_checkpoint('superdropnet_final.ckpt')

100%|██████████| 1.01M/1.01M [00:00<00:00, 9.35MiB/s]


In [ ]:
ckpt = torch.load('superdropnet_final.ckpt', map_location=torch.device('cpu'))
for key, val in ckpt['state_dict'].items():
    print(key, val.shape)
print()

model.layers.0.weight torch.Size([200, 9])
model.layers.0.bias torch.Size([200])
model.layers.1.weight torch.Size([200, 200])
model.layers.1.bias torch.Size([200])
model.layers.2.weight torch.Size([200, 200])
model.layers.2.bias torch.Size([200])
model.layers.3.weight torch.Size([4, 200])
model.layers.3.bias torch.Size([4])



/tmp/ipykernel_88456/3261250732.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('../trained_models/superdropnet/superdropnet_final.ckpt', map_location=

# Import WrappedSuperdropNet Class that Enforces Physical Constraints

In [1]:
timestep=60

In [8]:
from superdropnet import WrappedSuperdropNet

In [ ]:
wrappedNet = WrappedSuperdropNet(pretrained_path='superdropnet_final.ckpt', timestep=timestep)

/work/ka1176/paul/deep-subgrid/notebooks/../src/cloud_microphysics/superdropnet/superdropnet.py:148: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_ckpt = torch.load(se

In [10]:
# tmp
test_moments = torch.tensor([
                            [0, 1e-11, 1e-3, 1e3], # 0
                             [0, 1e-11, 1e-5, 1e3], # 0
                             [1e-3, 1e3, 1e-3, 1e3], # 1
                             [1e-3, 1e3, 1e-5, 1e3], # 1
                             [1e-3, 1e4, 0, 0], # 1
                             [1e-5, 1e5, 0, 0], 
                             [1e-11, 0, 1e-3, 1e3],
                             [2.7591e-05, 1.9099e+07, 2.2527e-06, 2.4127e+02],
                            [1.1215e-04, 2.0851e+07, 3.1428e-08, 9.0291e+01],
                            [1e-10,1e1,0,1e5]
                            ]).to(torch.float32).unsqueeze(1) # 0
#test_moments = torch.tensor([[[0, 1e-11, 1e-3, 1e3]]]).to(torch.float32)
results = wrappedNet(test_moments)
print(results)

tensor([[[0.0000e+00, 1.0000e-11, 1.0000e-03, 9.6336e+02]],

        [[0.0000e+00, 1.0000e-11, 1.0000e-05, 1.1199e+03]],

        [[1.0000e-03, 0.0000e+00, 1.0000e-03, 0.0000e+00]],

        [[1.0000e-03, 0.0000e+00, 1.0000e-05, 0.0000e+00]],

        [[1.0000e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[4.1914e-06, 1.0000e+05, 5.8086e-06, 0.0000e+00]],

        [[1.0000e-11, 0.0000e+00, 1.0000e-03, 0.0000e+00]],

        [[2.2731e-05, 1.8761e+07, 7.1125e-06, 0.0000e+00]],

        [[1.0739e-04, 2.0851e+07, 4.7947e-06, 0.0000e+00]],

        [[1.0000e-10, 1.0000e+01, 0.0000e+00, 1.0000e+05]]])


# Time Model Speed

This timining is not totally consistent. You get differences of 100us for the same model, which is like 20% of the total time. Be careful when benchmarking!

In [12]:
%%timeit
random_output = wrappedNet(random_input)

713 μs ± 26.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
import timeit

# warm up once if needed
_ = wrappedNet(random_input)

loops = 20000

# measure 200 runs with the same inputs
elapsed = timeit.timeit(lambda: wrappedNet(random_input), number=loops)
print(f"avg per run: {elapsed / loops:.6f} s")

avg per run: 0.000578 s


# Save as TorchScript

In [13]:
scripted_model = torch.jit.script(wrappedNet)

In [ ]:
print(scripted_model.code)
filename = 'superdropnet_final_sharma2025_60s.pt'
if os.path.exists(filename):
    print('Will not overwrite: ', filename)
else:
    scripted_model.save(filename)

def forward(self,
    all_moments_in: Tensor) -> Tensor:
  dim_i, dim_k, dim_m, = torch.size(all_moments_in)
  all_moments_in0 = torch.reshape(all_moments_in, [-1, dim_m])
  inputs = (self).create_input(all_moments_in0, )
  backbone = self.backbone
  predictions_updates = (backbone).test_step(inputs, )
  updates_std = self.updates_std
  _0 = torch.mul(predictions_updates, updates_std)
  updates_mean = self.updates_mean
  updates = torch.add(_0, updates_mean)
  _1 = torch.slice(torch.slice(all_moments_in0), 1, None, 4)
  timestep = self.timestep
  moments_out = torch.add(_1, torch.mul(updates, timestep))
  moments_out0 = (self).check_preds(moments_out, all_moments_in0, )
  moments_out1 = torch.reshape(moments_out0, [dim_i, dim_k, dim_m])
  return moments_out1



In [13]:
class IdentityModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return x

In [ ]:
idm = IdentityModel()
sidm = torch.jit.script(idm)
sidm.save('identity_model.pt')